# Alpha Shapes: From Point Clouds to Geometry

This notebook revisits `matlab/alpha-shapes/AlphaShapes.m` in pure Python, with a focus on intuition, equations, and interactive exploration.

## Mathematical idea

Given a finite point cloud $P\subset\mathbb{R}^2$, the **alpha complex** is a filtered subcomplex of the Delaunay triangulation.

A triangle is kept when the radius $R$ of its empty circumcircle is below a threshold:

$$
R \leq \alpha.
$$

As $\alpha$ grows, the reconstruction transitions from disconnected fragments to the convex hull. The boundary of the alpha shape is made of edges incident to exactly one retained triangle.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection, LineCollection
from scipy.spatial import Delaunay
from ipywidgets import interact, FloatLogSlider, IntSlider

plt.rcParams['figure.dpi'] = 120
rng = np.random.default_rng(7)

## 1) Build a non-convex toy point cloud

In [ ]:
def make_cloud(n=450, noise=0.018):
    # Crescent-like shape
    t = rng.uniform(0, 2 * np.pi, n)
    r = 0.35 + 0.12 * rng.standard_normal(n)
    x = 0.52 + (0.95 + 0.30 * np.cos(2 * t)) * r * np.cos(t)
    y = 0.52 + (0.65 + 0.15 * np.sin(3 * t)) * r * np.sin(t)
    pts = np.c_[x, y]
    pts += noise * rng.standard_normal(pts.shape)
    return np.clip(pts, 0.02, 0.98)

points = make_cloud()

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(points[:, 0], points[:, 1], s=7, color='black', alpha=0.8)
ax.set_aspect('equal')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('Input point cloud')
plt.show()

## 2) Delaunay triangulation and circumradius filter

In [ ]:
tri = Delaunay(points)
simplices = tri.simplices

A = points[simplices[:, 0]]
B = points[simplices[:, 1]]
C = points[simplices[:, 2]]

a = np.linalg.norm(B - C, axis=1)
b = np.linalg.norm(C - A, axis=1)
c = np.linalg.norm(A - B, axis=1)
s = 0.5 * (a + b + c)
area = np.sqrt(np.clip(s * (s - a) * (s - b) * (s - c), 1e-15, None))
R = (a * b * c) / (4.0 * area)  # circumradius

print('Number of Delaunay triangles:', len(simplices))
print('Circumradius range:', float(R.min()), float(R.max()))

In [ ]:
def alpha_shape_geometry(points, simplices, radii, alpha):
    keep = radii <= alpha
    kept = simplices[keep]

    edge_count = {}
    for t in kept:
        e = [(t[0], t[1]), (t[1], t[2]), (t[2], t[0])]
        for i, j in e:
            key = tuple(sorted((int(i), int(j))))
            edge_count[key] = edge_count.get(key, 0) + 1

    boundary = [k for k, v in edge_count.items() if v == 1]
    interior = [k for k, v in edge_count.items() if v == 2]
    return kept, boundary, interior

def euler_stats(points, simplices, radii, alpha):
    kept, boundary, interior = alpha_shape_geometry(points, simplices, radii, alpha)
    if len(kept) == 0:
        return {'V': 0, 'E': 0, 'F': 0, 'chi': 0}
    verts = np.unique(kept.ravel())
    edges = len(boundary) + len(interior)
    faces = len(kept)
    chi = len(verts) - edges + faces
    return {'V': len(verts), 'E': edges, 'F': faces, 'chi': chi}

## 3) Interactive alpha filtering

In [ ]:
def show_alpha(alpha=0.06):
    kept, boundary, _ = alpha_shape_geometry(points, simplices, R, alpha)
    stats = euler_stats(points, simplices, R, alpha)

    fig, ax = plt.subplots(figsize=(6.2, 6.2))
    if len(kept) > 0:
        polys = [points[t] for t in kept]
        coll = PolyCollection(polys, facecolor=(0.2, 0.6, 0.95, 0.18), edgecolor='none')
        ax.add_collection(coll)

    if boundary:
        segs = [points[list(e)] for e in boundary]
        lc = LineCollection(segs, colors='crimson', linewidths=2.0)
        ax.add_collection(lc)

    ax.scatter(points[:, 0], points[:, 1], s=7, color='black', alpha=0.8)
    ax.set_aspect('equal')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(
        f'alpha={alpha:.4f} | V={stats["V"]}, E={stats["E"]}, F={stats["F"]}, '
        f'chi={stats["chi"]}'
    )
    plt.show()

interact(show_alpha, alpha=FloatLogSlider(base=10, min=-3, max=-0.6, step=0.02, value=0.06));

## 4) Evolution over a range of alpha values

In [ ]:
alphas = np.quantile(R, np.linspace(0.05, 0.75, 8))

fig, axes = plt.subplots(2, 4, figsize=(13, 6.5), constrained_layout=True)
for ax, alpha in zip(axes.flat, alphas):
    kept, boundary, _ = alpha_shape_geometry(points, simplices, R, alpha)
    if len(kept) > 0:
        polys = [points[t] for t in kept]
        ax.add_collection(PolyCollection(polys, facecolor=(0.4, 0.75, 1.0, 0.20), edgecolor='none'))
    if boundary:
        segs = [points[list(e)] for e in boundary]
        ax.add_collection(LineCollection(segs, colors='darkred', linewidths=1.2))
    ax.scatter(points[:, 0], points[:, 1], s=4, color='black', alpha=0.5)
    ax.set_aspect('equal')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(f'alpha={alpha:.3f}')
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

## 5) Key takeaways

- Alpha shapes are a geometric multiscale model between point cloud and hull.
- The Delaunay triangulation provides an efficient combinatorial scaffold.
- The threshold $\alpha$ controls topological simplification.